# FASTA Sequence Analyzer

#### This notebook reads and analyzes FASTA files

Features:
- Read FASTA files using BioPython
- Count nucleotides
- Visualize results using matplotlib

Author: Pong Somboon

In [ ]:
%pip install biopython matplotlib 

In [ ]:
from Bio import SeqIO
import os
import matplotlib.pyplot as plt
import numpy as np

print("Libraries imported successfully.")

In [ ]:
# FASTA file reading function

def read(file):
    """
    Reads a FASTA file and returns a dictionary with each sequence.
    """
    sequences = {}
    for record in SeqIO.parse(file, "fasta"):
        sequences[record.id] = str(record.seq)
    return sequences

print("Defined read() successfully.")

In [ ]:
# Base counting function

def base_count(sequence):
    """
    Counts occurence of each base.
    """
    sequence = sequence.upper()

    return {
        'A': sequence.count('A'),
        'T': sequence.count('T'),
        'G': sequence.count('G'),
        'C': sequence.count('C')
    }

print("Defined base_count() successfully.")

In [ ]:
# GC content function

def gc_content(sequence):
    """
    Calculates the GC content.
    """
    sequence = sequence.upper()
    G_count = sequence.count('G')
    C_count = sequence.count('C')
    total_count = len(sequence)

    if total_count == 0:
        return 0.0
    
    return (G_count + C_count) / total_count * 100

print("Defined gc_content() successfully.")


---

## Load FASTA File

In [ ]:
#Input FASTA file name

filename = input("Enter FASTA file name (e.g. abc.fasta): ").strip()

# Validate file exists

while not os.path.exists(filename):
    print(f"Error: File '{filename}' not found.")
    print(f"Please make sure the file is in the current directory: {os.getcwd()}")

    user_input = input("Try again? (y/n): ").strip().lower()
    if user_input in ['y', 'yes']:
        filename = input("Enter FASTA file name (e.g. abc.fasta): ").strip()
    else:
        print("Exiting program.")
        raise FileNotFoundError(f"File '{filename}' not found.")

#Load sequences from the FASTA file

try:
    sequences = read(filename)
    print(f"Successfully read {len(sequences)} sequences from '{filename}'.")
except Exception as e:
    print(f"Error reading file '{filename}': {e}")
    raise


---

## Sequence Analysis

### Statistics for each sequence:

In [ ]:
# Base composition, GC content, sequence length, and preview for each sequence

for seq_id, seq in sequences.items():
    print("=" * 40)
    print(f"Sequence ID: {seq_id}")
    print("-" * 40)
    print(f"Length: {len(seq):,} bp")
    print(f"GC Content: {gc_content(seq):.2f}%")
    
    print(f"\nBase composition:")
    for base, count in base_count(seq).items():
        percentage = (count / len(seq)) * 100 if len(seq) > 0 else 0
        print(f"  {base}: {count:5,} ({percentage:5.2f}%)")

    print(f"\nSequence preview (up to 50 bp): {seq[:50]}{'...' if len(seq) > 50 else ''}")       


---

## Visualizations

### GC Content across sequences:


In [ ]:
headers = list(sequences.keys())
gc_values = [gc_content(seq) for seq in sequences.values()]

# Create bar chart

plt.figure(figsize=(12, 6))
bars = plt.bar(headers, gc_values, color='skyblue', edgecolor='black', linewidth=1.5)

# Add value labels on top of bars
for bar, value in zip(bars, gc_values):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height,
                f'{value:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add labels and title

plt.xlabel('Sequence ID', fontsize=13)
plt.ylabel('GC Content (%)', fontsize=13)
plt.title('GC Content of Sequences', fontsize=15, fontweight='bold', pad=20)
plt.ylim(0, 100)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


### Nucleotide Composition

In [ ]:
# Assigning colors for each base
bases = ['A', 'T', 'G', 'C']
colors = ['#FF9999', '#99FF99', '#9999FF', '#FFCC99']

# Percentages of each base, changes it to array so it can be transposed
data = []
for seq in sequences.values():
    counts = base_count(seq)
    total = len(seq)
    percentages = [(counts[base] / total * 100) if total > 0 else 0 for base in bases]
    data.append(percentages)

data = np.array(data).T

# Create stacked bar chart

fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(len(sequences))

for i, base in enumerate(bases):
    ax.bar(headers, data[i], bottom=bottom, color=colors[i], edgecolor='black', linewidth=1.5, label=base)
    bottom += data[i]

ax.set_xlabel('Sequence ID', fontsize=13)
ax.set_ylabel('Base Composition (%)', fontsize=13)
ax.set_title('Base Composition of Sequences', fontsize=15, fontweight='bold', pad=20)
ax.set_ylim(0, 100)
ax.legend(title='Base', fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()
plt.show()


---

### Overview of all sequences:

In [ ]:
print(f"\nTotal sequences analyzed: {len(sequences)}")
print("\nSequence lengths:")
print(f"Minimum: {min(len(seq) for seq in sequences.values()):,} bp")
print(f"Maximum: {max(len(seq) for seq in sequences.values()):,} bp")
print(f"Average: {sum(len(seq) for seq in sequences.values()) / len(sequences):,.2f} bp")
print(f"Total: {sum(len(seq) for seq in sequences.values()):,} bp")

print("\nGC Content:")
print(f"Minimum: {min(gc_content(seq) for seq in sequences.values()):.2f}%")
print(f"Maximum: {max(gc_content(seq) for seq in sequences.values()):.2f}%")
print(f"Average: {sum(gc_content(seq) for seq in sequences.values()) / len(sequences):.2f}%")